# Backend experiment: LangGraph Platform self-hosted (`langgraph dev`)

LangGraph Platform's real Docker path (`langgraph up`) needs an Enterprise license key, which is impractical for a free pilot. This backend instead runs `langgraph dev`: a license-free, in-memory server that exposes the identical assistants/threads/runs REST API, as a local subprocess instead of a container.

This notebook is a self-contained, reproducible experiment: it starts the `langgraph-platform` backend, deploys the three example agents (`researcher`, `coder`, `reviewer`), calls each one directly through `langgraph_sdk`, then runs the `supervisor` agent locally and has it orchestrate all three **via `RemoteGraph`**.


In [1]:
import sys

sys.path.insert(0, "/Users/jyje/repo/jyje/pilot-langchain-remotegraph")

from langgraph_sdk import get_sync_client

from remotegraph.backends.langgraph_platform import LangGraphPlatformBackend as Backend

backend = Backend()
print(backend.name, "->", backend.base_url)


langgraph-platform -> http://127.0.0.1:2024


## 1. Start the backend

`backend.up()` spawns `uv run langgraph dev --port 2024 --no-browser` as a background subprocess (no Docker).

In [2]:
SERVED_AGENTS = {
    "researcher": "agents/researcher/graph.py:graph",
    "coder": "agents/coder/graph.py:graph",
    "reviewer": "agents/reviewer/graph.py:graph",
}
backend.deploy(SERVED_AGENTS)
backend.up()
print(backend.status())


running


## 2. List registered assistants

In [3]:
import time

client = get_sync_client(url=backend.base_url)

for _attempt in range(10):
    try:
        assistants = list(client.assistants.search())
        if assistants:
            break
    except Exception as exc:
        print("waiting for backend...", exc)
    time.sleep(3)

for a in assistants:
    print(a["assistant_id"], "graph_id=" + str(a.get("graph_id")))


ca7018db-539a-5a5f-b9c2-8622330bec7d graph_id=reviewer
c926ac5a-b04e-5949-878a-8e4830d4338b graph_id=researcher
ad9aa870-1e45-562b-a83d-73b96694ea13 graph_id=coder


## 3. Call each agent directly

In [4]:
def call(name: str, message: str) -> str:
    last = None
    stream = client.runs.stream(
        None, name, input={"messages": [{"role": "user", "content": message}]}, stream_mode="values"
    )
    for chunk in stream:
        if chunk.event == "values" and chunk.data.get("messages"):
            last = chunk.data["messages"][-1]["content"]
    return last


print(call("coder", "What is 13 * 7? Just the number."))


21


In [5]:
print(call("researcher", "What is LangGraph in one sentence?"))


LangGraph is an open-source framework within LangChain that allows developers to build and manage complex, stateful AI agent workflows by structuring them as a directed graph of interconnected nodes (actions) and edges (flow control).


In [6]:
print(call("reviewer", "Review this code: def add(a, b): return a + b"))


The code is correct and functional.

**Verdict:** Passed.


## 4. Run the supervisor pipeline via RemoteGraph

The supervisor never imports `researcher`/`coder`/`reviewer` directly -- it only knows the backend's base URL and calls them through `langgraph.pregel.remote.RemoteGraph`, exactly as it would for a real LangGraph Platform deployment.

In [7]:
import os

os.environ["REMOTEGRAPH_BASE_URL"] = backend.base_url

sys.path.insert(0, "/Users/jyje/repo/jyje/pilot-langchain-remotegraph")
from agents.supervisor.graph import graph as supervisor_graph

task = "Write a one-line python function that returns the square of a number."
result = supervisor_graph.invoke({"task": task})
print("--- research ---")
print(result["research"])
print()
print("--- code ---")
print(result["code"])
print()
print("--- review ---")
print(result["review"])


--- research ---
```python
square = lambda x: x**2
```

**Usage examples:**

```python
print(square(5))  # Output: 25
print(square(-3)) # Output: 9
```

--- code ---
```python
square = lambda x: x**2
```

--- review ---
**Verdict:** Mostly Correct, but Stylistically Flawed

The code is functionally correct. It successfully creates a function that squares its input `x`. However, using a simple assignment for a multi-line or complex function body is usually better served by `def`.

**Issues Found:**

1.  **Avoid Lambda Assignment (Style/Readability):** While this specific use case is very short, assigning a `lambda` expression to a variable name like `square` is generally discouraged in favor of using a standard function definition (`def`). This improves readability and maintainability, especially for those new to Python or reading stack traces.

**Suggestion:**

For improved readability and adherence to standard Python style guides (PEP 8), use a `def` block instead:

```python
def squa

## 5. Tear down

In [8]:
backend.down()
print(backend.status())


stopped
